### Valentin Lorenzo Carroza Granifo

### 22.437.518-2

In [1]:
import numpy as np
import pandas as pd

In [2]:
from pathlib import Path
import csv
import json
import pickle
import importlib.util

In [3]:
archivo = Path.cwd() / "catalogo_estelar.csv"

In [17]:
class CatalogoEstelar:
    """Catálogo de estrellas almacenado como un arreglo de NumPy por columna.

    Al construirse, lee el CSV, guarda cada columna como atributo y elimina
    automáticamente los registros inválidos. Además ofrece un resumen
    estadístico, métodos para guardar y recuperar los datos en cuatro
    formatos (CSV, NPY, NPZ y Pickle) y métodos para verificarlos y compararlos.
    """

    # Nombres de las columnas del CSV. Se definen una sola vez para recorrerlos
    # en bucles y garantizar que todas las columnas reciban el mismo tratamiento.
    COLUMNAS = ("id", "nombre", "temperatura_K", "luminosidad_Lsol", "distancia_pc")

    # Directorio donde se guardan los archivos de salida. Es una ruta relativa:
    # se crea dentro de la carpeta de trabajo del notebook (la de Path.cwd()).
    DIRECTORIO = Path("resultados")

    # =====================================================================
    # CONSTRUCTOR
    # =====================================================================
    def __init__(self, archivo):
        """Lee el CSV, guarda cada columna como arreglo y ejecuta la limpieza."""
        # Se convierte a Path para tratar de la misma forma una ruta recibida
        # como texto o como objeto Path.
        self.archivo = Path(archivo)

        # genfromtxt lee el CSV:
        #   names=True -> la primera fila se usa como encabezado, de modo que
        #                 cada columna se obtiene por su nombre.
        #   dtype=None -> NumPy detecta el tipo de cada columna (texto o número).
        datos = np.genfromtxt(self.archivo, delimiter=",", names=True,
                              dtype=None, encoding="utf-8")

        # Cada columna del CSV se guarda como un arreglo, en un atributo con su
        # mismo nombre (self.id, self.nombre, self.temperatura_K, etc.).
        for columna in self.COLUMNAS:
            setattr(self, columna, datos[columna])

        # El enunciado exige que la limpieza se ejecute automáticamente aquí,
        # aunque debe seguir siendo un método independiente.
        self.limpiar_datos()

    # =====================================================================
    # LIMPIEZA DE DATOS
    # =====================================================================
    def limpiar_datos(self):
        """Elimina de todas las columnas los registros inválidos.

        Un registro es inválido si:
          - tiene valores ausentes o no finitos (nan, inf, -inf);
          - tiene valores no positivos en temperatura, luminosidad o distancia;
          - su identificador ya apareció antes (se conserva solo la primera).

        Devuelve la cantidad de registros eliminados. Si se llama de nuevo sobre
        datos ya limpios, no modifica nada y devuelve 0.
        """
        n = len(self.id)  # número de registros actuales
        fisicas = [self.temperatura_K, self.luminosidad_Lsol, self.distancia_pc]

        # Máscara booleana: True en la posición i significa "el registro i es
        # inválido". Se parte de todos False (ningún registro inválido) y se
        # acumulan los criterios con el operador |= (OR lógico).
        invalido = np.zeros(n, dtype=bool)

        # --- Criterio (a): valores no finitos o no positivos ----------------
        for columna in fisicas:
            # np.isfinite es False para nan, inf y -inf; el ~ invierte el
            # resultado para marcar justamente esos casos.
            invalido |= ~np.isfinite(columna)
            # Las propiedades físicas deben ser estrictamente positivas.
            # (nan <= 0 es False, pero el nan ya fue marcado en la línea anterior.)
            invalido |= (columna <= 0)

        # Si los identificadores son numéricos con decimales, un id ausente
        # aparece como nan. Esta comprobación se omite si el id es entero o texto.
        if np.issubdtype(self.id.dtype, np.floating):
            invalido |= ~np.isfinite(self.id)

        # --- Criterios (a) y (c): identificadores duplicados ----------------
        # np.unique con return_index=True devuelve, para cada id distinto, la
        # posición de su PRIMERA aparición.
        _, primeras = np.unique(self.id, return_index=True)

        # Máscara que es True solo en esas primeras apariciones.
        es_primera = np.zeros(n, dtype=bool)
        es_primera[primeras] = True

        # Todo registro que no sea la primera aparición de su id es un
        # duplicado posterior y se marca como inválido.
        invalido |= ~es_primera

        # --- Criterio (b): eliminación coherente en todas las columnas ------
        # Se usa UNA SOLA máscara para las cinco columnas. Así todos los
        # arreglos conservan la misma longitud y la fila i sigue correspondiendo
        # a la misma estrella en cada uno.
        validos = ~invalido
        if invalido.any():
            for columna in self.COLUMNAS:
                setattr(self, columna, getattr(self, columna)[validos])

        # Cantidad de registros eliminados. En una segunda llamada será 0, lo
        # que demuestra que el método no altera registros ya válidos.
        return int(invalido.sum())

    # =====================================================================
    # ANÁLISIS CON NUMPY
    # =====================================================================
    def resumen(self):
        """Calcula y presenta las propiedades globales del catálogo limpio.

        Usa solo operaciones vectorizadas de NumPy (sin recorrer estrellas).
        Devuelve un diccionario con los resultados, además de imprimirlos.
        """
        n = self.id.size

        # Un catálogo vacío no admite max, min ni argmax: se evita el error.
        if n == 0:
            print("El catálogo no contiene registros válidos.")
            return {}

        # (a) Número total, temperatura media y temperatura máxima.
        temp_media = np.mean(self.temperatura_K)
        temp_max = np.max(self.temperatura_K)

        # (b) Luminosidad mediana y distancias mínima y máxima.
        lum_mediana = np.median(self.luminosidad_Lsol)
        dist_min = np.min(self.distancia_pc)
        dist_max = np.max(self.distancia_pc)

        # (c) argmax/argmin entregan la POSICIÓN del extremo; esa posición se
        # usa como índice en 'nombre' para obtener la estrella. Funciona porque
        # la limpieza mantiene la correspondencia entre todas las columnas.
        mas_caliente = self.nombre[np.argmax(self.temperatura_K)]
        mas_cercana = self.nombre[np.argmin(self.distancia_pc)]

        # Los resultados se devuelven en un diccionario para poder reutilizarlos
        # en otras celdas (por ejemplo, resultados["temperatura_media_K"]).
        resultados = {
            "n_estrellas": n,
            "temperatura_media_K": temp_media,
            "temperatura_maxima_K": temp_max,
            "luminosidad_mediana_Lsol": lum_mediana,
            "distancia_minima_pc": dist_min,
            "distancia_maxima_pc": dist_max,
            "estrella_mas_caliente": mas_caliente,
            "estrella_mas_cercana": mas_cercana,
        }

        # Presentación clara: cada propiedad con su nombre y su unidad
        # (K = kelvin, L☉ = luminosidades solares, pc = parsecs).
        print("=" * 50)
        print("RESUMEN DEL CATÁLOGO ESTELAR")
        print("=" * 50)
        print(f"Número total de estrellas    : {n}")
        print(f"Temperatura media            : {temp_media:.2f} K")
        print(f"Temperatura máxima           : {temp_max:.2f} K")
        print(f"Luminosidad mediana          : {lum_mediana:.4g} L☉")
        print(f"Distancia mínima             : {dist_min:.4g} pc")
        print(f"Distancia máxima             : {dist_max:.4g} pc")
        print(f"Estrella de mayor temperatura: {mas_caliente}")
        print(f"Estrella más cercana         : {mas_cercana}")
        print("=" * 50)
        return resultados

    # =====================================================================
    # LECTURA Y ESCRITURA DE ARCHIVOS
    # =====================================================================
    @classmethod
    def _preparar_directorio(cls):
        """Crea el directorio de destino si no existe y devuelve su ruta.

        Cada método de guardado lo llama por sí mismo, de modo que ninguno
        depende de que otro haya creado antes la carpeta (requisito del
        enunciado).
        """
        # parents=True -> crea también las carpetas intermedias si hicieran falta.
        # exist_ok=True -> si el directorio ya existe, no produce error.
        cls.DIRECTORIO.mkdir(parents=True, exist_ok=True)
        return cls.DIRECTORIO

    # --- (a) Formato CSV ----------------------------------------------------
    def guardar_csv(self):
        """Guarda todas las columnas, con encabezados, en catalogo_copia.csv."""
        ruta = self._preparar_directorio() / "catalogo_copia.csv"
        # newline="" evita líneas en blanco extra en Windows;
        # encoding="utf-8" conserva correctamente tildes y caracteres especiales.
        with open(ruta, "w", newline="", encoding="utf-8") as f:
            escritor = csv.writer(f)
            # Primera fila: los encabezados (nombres de las columnas).
            escritor.writerow(self.COLUMNAS)
            # zip(*columnas) reúne el i-ésimo elemento de cada columna, por lo
            # que cada tupla resultante es una fila completa (una estrella).
            escritor.writerows(zip(*(getattr(self, c) for c in self.COLUMNAS)))
        return ruta

    @classmethod
    def leer_csv(cls):
        """Relee catalogo_copia.csv y devuelve un nuevo CatalogoEstelar."""
        # El enunciado pide releer el archivo con el constructor de la clase.
        # Como el constructor ya lee un CSV con encabezados, basta pasarle la ruta.
        return cls(cls.DIRECTORIO / "catalogo_copia.csv")

    # --- (b) Formato NPY ----------------------------------------------------
    def guardar_npy(self):
        """Guarda solo las temperaturas en temperaturas.npy."""
        ruta = self._preparar_directorio() / "temperaturas.npy"
        # np.save almacena un único arreglo en formato binario de NumPy.
        np.save(ruta, self.temperatura_K)
        return ruta

    @classmethod
    def leer_npy(cls):
        """Recupera el arreglo de temperaturas guardado en temperaturas.npy."""
        return np.load(cls.DIRECTORIO / "temperaturas.npy")

    # --- (c) Formato NPZ ----------------------------------------------------
    def guardar_npz(self):
        """Guarda todas las columnas, comprimidas, en catalogo_completo.npz."""
        ruta = self._preparar_directorio() / "catalogo_completo.npz"
        # Diccionario {nombre de columna: arreglo}; savez_compressed guarda
        # cada arreglo bajo su nombre dentro del mismo archivo.
        arreglos = {c: getattr(self, c) for c in self.COLUMNAS}
        # Los nombres se convierten a texto de NumPy (dtype unicode, no objeto).
        # Los arreglos de objetos exigirían allow_pickle=True al leerlos;
        # los de texto no.
        arreglos["nombre"] = arreglos["nombre"].astype(str)
        np.savez_compressed(ruta, **arreglos)
        return ruta

    @classmethod
    def leer_npz(cls):
        """Recupera todas las columnas de catalogo_completo.npz como diccionario."""
        # np.load sobre un .npz devuelve un objeto de lectura diferida: los
        # arreglos se cargan al accederlos, por lo que deben extraerse mientras
        # el archivo sigue abierto (dentro del bloque with).
        with np.load(cls.DIRECTORIO / "catalogo_completo.npz") as datos:
            return {c: datos[c] for c in cls.COLUMNAS}

    # --- (d) Formato Pickle -------------------------------------------------
    def guardar_pickle(self):
        """Guarda un diccionario con todas las columnas en catalogo_completo.pkl."""
        ruta = self._preparar_directorio() / "catalogo_completo.pkl"
        diccionario = {c: getattr(self, c) for c in self.COLUMNAS}
        # "wb" = escritura en modo binario, requerida por pickle.
        with open(ruta, "wb") as f:
            pickle.dump(diccionario, f)
        return ruta

    @classmethod
    def leer_pickle(cls):
        """Recupera el diccionario guardado en catalogo_completo.pkl."""
        # ADVERTENCIA DE SEGURIDAD: pickle.load puede ejecutar código arbitrario
        # contenido en el archivo. Solo debe abrirse un archivo creado por uno
        # mismo o proveniente de una fuente confiable.
        with open(cls.DIRECTORIO / "catalogo_completo.pkl", "rb") as f:
            return pickle.load(f)

    # =====================================================================
    # VERIFICACIÓN Y COMPARACIÓN DE FORMATOS
    # =====================================================================
    def _comparar_con_original(self, datos):
        """Compara un diccionario {columna: arreglo} con el catálogo actual.

        Acepta diccionarios parciales (el NPY solo trae las temperaturas).
        Devuelve {comprobación: bool}: una entrada para el número de
        estrellas y otra por cada columna recibida.
        """
        # Solo se evalúan las columnas que el formato realmente almacena.
        columnas = [c for c in self.COLUMNAS if c in datos]

        # Número de estrellas recuperadas frente al del catálogo limpio.
        resultados = {"n_estrellas": len(datos[columnas[0]]) == len(self.id)}

        for columna in columnas:
            original = getattr(self, columna)
            recuperado = datos[columna]

            # Si las formas difieren no tiene sentido comparar valores
            # (además, allclose podría fallar al intentar hacer "broadcast").
            if recuperado.shape != original.shape:
                resultados[columna] = False
            elif columna in ("id", "nombre"):
                # Identificadores y nombres: igualdad exacta.
                resultados[columna] = bool(np.array_equal(recuperado, original))
            else:
                # Valores numéricos: igualdad dentro de la tolerancia numérica.
                resultados[columna] = bool(np.allclose(recuperado, original))
        return resultados

    def verificar_archivos(self):
        """Lee los archivos generados y verifica que coincidan con el catálogo.

        Para cada formato comprueba el número de estrellas y los valores de las
        columnas que almacena. Devuelve un diccionario {formato: resultados};
        el valor es None si el archivo no existe.
        """
        # Cada lector devuelve un diccionario {columna: arreglo}, de modo que
        # todos los formatos se comparan con el mismo procedimiento.
        def datos_csv():
            # El CSV se relee mediante el constructor, que devuelve un catálogo.
            copia = self.leer_csv()
            return {c: getattr(copia, c) for c in self.COLUMNAS}

        def datos_npy():
            # El NPY contiene únicamente las temperaturas.
            return {"temperatura_K": self.leer_npy()}

        lectores = {
            "CSV": datos_csv,
            "NPY": datos_npy,
            "NPZ": self.leer_npz,
            "Pickle": self.leer_pickle,
        }

        resultados = {}
        for formato, lector in lectores.items():
            try:
                resultados[formato] = self._comparar_con_original(lector())
            except FileNotFoundError:
                # Si el archivo no existe se informa, en lugar de detener todo.
                resultados[formato] = None

        # --- Presentación de los resultados ---------------------------------
        print("=" * 60)
        print("VERIFICACIÓN DE LOS ARCHIVOS GENERADOS")
        print("=" * 60)
        todo_coincide = True
        for formato, comprobaciones in resultados.items():
            print(f"\nFormato {formato}")
            if comprobaciones is None:
                print("  Archivo no encontrado (ejecute antes el método guardar_ correspondiente).")
                todo_coincide = False
                continue
            for nombre, coincide in comprobaciones.items():
                print(f"  {nombre:<16}: {'coincide' if coincide else 'NO coincide'}")
            todo_coincide = todo_coincide and all(comprobaciones.values())

        print("\n" + "=" * 60)
        if todo_coincide:
            print("Resultado: los datos recuperados coinciden con los originales.")
        else:
            print("Resultado: se detectaron diferencias o archivos faltantes.")
        print("=" * 60)
        return resultados

    def comparar_formatos(self):
        """Determina el tamaño en bytes de cada archivo y lo muestra en una tabla.

        Devuelve un diccionario {formato: tamaño en bytes}; el valor es None
        si el archivo no existe.
        """
        # formato: (nombre del archivo, contenido almacenado)
        archivos = {
            "CSV": ("catalogo_copia.csv", "Catálogo completo"),
            "NPY": ("temperaturas.npy", "Solo temperaturas"),
            "NPZ": ("catalogo_completo.npz", "Catálogo completo"),
            "Pickle": ("catalogo_completo.pkl", "Catálogo completo"),
        }

        tamanos = {}
        print("=" * 72)
        print(f"{'Formato':<8}{'Archivo':<24}{'Contenido':<20}{'Tamaño (bytes)':>16}")
        print("-" * 72)
        for formato, (nombre, contenido) in archivos.items():
            ruta = self.DIRECTORIO / nombre
            # Path.stat().st_size entrega el tamaño del archivo en bytes.
            tamano = ruta.stat().st_size if ruta.exists() else None
            tamanos[formato] = tamano
            texto = f"{tamano:d}" if tamano is not None else "no existe"
            print(f"{formato:<8}{nombre:<24}{contenido:<20}{texto:>16}")
        print("=" * 72)
        return tamanos

In [20]:
archivo = Path.cwd() / "catalogo_estelar.csv"
catalogo = CatalogoEstelar(archivo)

catalogo.guardar_csv()
catalogo.guardar_npy()
catalogo.guardar_npz()
catalogo.guardar_pickle()

catalogo.verificar_archivos()
catalogo.comparar_formatos()

VERIFICACIÓN DE LOS ARCHIVOS GENERADOS

Formato CSV
  n_estrellas     : coincide
  id              : coincide
  nombre          : coincide
  temperatura_K   : coincide
  luminosidad_Lsol: coincide
  distancia_pc    : coincide

Formato NPY
  n_estrellas     : coincide
  temperatura_K   : coincide

Formato NPZ
  n_estrellas     : coincide
  id              : coincide
  nombre          : coincide
  temperatura_K   : coincide
  luminosidad_Lsol: coincide
  distancia_pc    : coincide

Formato Pickle
  n_estrellas     : coincide
  id              : coincide
  nombre          : coincide
  temperatura_K   : coincide
  luminosidad_Lsol: coincide
  distancia_pc    : coincide

Resultado: los datos recuperados coinciden con los originales.
Formato Archivo                 Contenido             Tamaño (bytes)
------------------------------------------------------------------------
CSV     catalogo_copia.csv      Catálogo completo               7046
NPY     temperaturas.npy        Solo temperaturas  

{'CSV': 7046, 'NPY': 864, 'NPZ': 3818, 'Pickle': 9240}

## Comparación de formatos

Tamaños (bytes): CSV = 7046, NPY = 864 (solo temperaturas), NPZ = 3818, Pickle = 9240.

El NPY no se compara con los otros porque solo guarda las temperaturas. Entre los que guardan el catálogo completo, el más pequeño fue el NPZ y el más grande el Pickle.

Diferencias:
- CSV: archivo de texto, se puede abrir en cualquier programa, pero no guarda los tipos de dato.
- NPY: guarda un solo arreglo de NumPy, es rápido y compacto.
- NPZ: guarda varios arreglos en un archivo y los comprime, por eso fue el más pequeño.
- Pickle: guarda objetos de Python, solo sirve en Python y no es seguro abrir archivos de otras personas.

¿Qué formato usaría para compartir el catálogo? \
CSV, porque cualquiera puede abrirlo y no necesita Python ni NumPy.

¿Cuál es mejor para guardar varios arreglos de NumPy en un solo archivo? \
NPZ, porque guarda varios arreglos con nombre en un solo archivo, los comprime y conserva su tipo y forma.